In [ ]:

import os
import json
import shutil
from google.colab import drive
import requests
from urllib.parse import urlparse, unquote
import time

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
kaggle_json = {
    "username":"",
    "key":""
}
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_json, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)

In [ ]:
SAVE_DIR = "/content/drive/MyDrive/Economic_Times/Dataset"
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# ==========================================================
# Dataset 6 - MITRE ATT&CK / STIX
# ==========================================================

import os
import shutil

BASE_DIR = os.path.join(SAVE_DIR, "MITRE-STIX")
GIT_DIR = os.path.join(BASE_DIR, ".git")

if not os.path.exists(BASE_DIR):
    print("Cloning MITRE ATT&CK repository...")
    !git clone https://github.com/mitre/cti.git "$BASE_DIR"

elif os.path.exists(GIT_DIR):
    print("MITRE ATT&CK repository already exists.")
    print("Updating...")
    !git -C "$BASE_DIR" pull

else:
    print("Folder exists but is not a Git repository.")
    print("Removing and cloning again...")
    shutil.rmtree(BASE_DIR)
    !git clone https://github.com/mitre/cti.git "$BASE_DIR"

print("\nDone!")

Cloning MITRE ATT&CK repository...
Cloning into '/content/drive/MyDrive/Economic_Times/Dataset/MITRE-STIX'...
remote: Enumerating objects: 686989, done.
remote: Counting objects: 100% (10095/10095), done.
remote: Compressing objects: 100% (5294/5294), done.
remote: Total 686989 (delta 9884), reused 4802 (delta 4801), pack-reused 676894 (from 5)
Receiving objects: 100% (686989/686989), 365.26 MiB | 11.15 MiB/s, done.
Resolving deltas: 100% (649478/649478), done.
Updating files: 100% (36418/36418), done.

Done!


In [ ]:
import os
import json
import glob
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. Configuration & Initializations
# ---------------------------------------------------------
MONGO_URI = "mongodb+srv://@rag..mongodb.net/?appName=RAG"
SAVE_DIR = "/content/drive/MyDrive/Economic_Times/Dataset"

BATCH_SIZE = 128  # Process in small batches to save RAM

print("Connecting to MongoDB Atlas...")
client = MongoClient(MONGO_URI)
db = client["cyber_intelligence_rag"]

# Collections
kev_col = db["cisa_kev_embeddings"]
mitre_col = db["mitre_attack_embeddings"]

print("Loading SentenceTransformer model (all-MiniLM-L6-v2)...")
# Lightweight model (~80MB), fast and memory-efficient
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device=device
)

print(f"Model loaded on: {next(model.parameters()).device}")


# Helper function to process in batches to save RAM
def batch_insert(collection, docs_batch):
    if not docs_batch:
        return

    # Generate embeddings for the batch text field
    texts = [doc["text_to_embed"] for doc in docs_batch]
    embeddings = model.encode(texts, show_progress_bar=False, batch_size=32)

    for doc, emb in zip(docs_batch, embeddings):
        doc["embedding"] = emb.tolist()

    collection.insert_many(docs_batch)

# ---------------------------------------------------------
# 2. Process CISA KEV Data
# ---------------------------------------------------------
def process_cisa_kev():
    kev_path = os.path.join(SAVE_DIR, "KVE", "known_exploited_vulnerabilities.json")
    if not os.path.exists(kev_path):
        print(f"Skipping KEV: File not found at {kev_path}")
        return

    print("\n--- Processing CISA KEV Feed ---")
    with open(kev_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    vulnerabilities = data.get("vulnerabilities", [])
    batch = []
    total_processed = 0

    for item in vulnerabilities:
        cve_id = item.get("cveID", "N/A")
        name = item.get("vulnerabilityName", "")
        desc = item.get("shortDescription", "")
        vendor = item.get("vendorProject", "")
        product = item.get("product", "")

        # Construct meaningful text representation for vector indexing
        text_content = f"CVE: {cve_id} | Product: {vendor} {product} | Name: {name} | Description: {desc}"

        doc = {
            "cveID": cve_id,
            "vendorProject": vendor,
            "product": product,
            "vulnerabilityName": name,
            "shortDescription": desc,
            "requiredAction": item.get("requiredAction"),
            "knownRansomwareCampaignUse": item.get("knownRansomwareCampaignUse"),
            "text_to_embed": text_content
        }

        batch.append(doc)
        if len(batch) >= BATCH_SIZE:
            batch_insert(kev_col, batch)
            total_processed += len(batch)
            print(f"KEV Processed: {total_processed} documents...", end="\r")
            batch = []

    # Insert remaining documents
    if batch:
        batch_insert(kev_col, batch)
        total_processed += len(batch)

    print(f"\nCompleted CISA KEV! Total documents stored: {total_processed}")

# ---------------------------------------------------------
# 3. Process MITRE ATT&CK STIX Data
# ---------------------------------------------------------
def process_mitre_stix():
    stix_dir = os.path.join(SAVE_DIR, "MITRE-STIX")
    if not os.path.exists(stix_dir):
        print(f"Skipping MITRE: Directory not found at {stix_dir}")
        return

    print("\n--- Processing MITRE ATT&CK STIX Feed ---")
    # Find all main domain json bundle files
    json_files = glob.glob(os.path.join(stix_dir, "**", "*.json"), recursive=True)

    batch = []
    total_processed = 0

    # Primary STIX types relevant for Threat Intelligence RAG
    TARGET_TYPES = {"attack-pattern", "malware", "tool", "intrusion-set", "course-of-action", "x-mitre-tactic"}

    for filepath in json_files:
        # Ignore git directory files or hidden files
        if ".git" in filepath:
            continue

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                stix_data = json.load(f)
        except Exception:
            continue

        objects = stix_data.get("objects", []) if isinstance(stix_data, dict) else []

        for obj in objects:
            obj_type = obj.get("type")

            # Filter out non-essential or structural objects (e.g., relationships)
            if obj_type not in TARGET_TYPES or obj.get("revoked", False):
                continue

            name = obj.get("name", "")
            desc = obj.get("description", "")
            if not desc:
                continue

            # Extract MITRE external ID (e.g., T1059)
            mitre_id = "N/A"
            for ref in obj.get("external_references", []):
                if ref.get("source_name") in ["mitre-attack", "mitre-mobile-attack", "mitre-ics-attack"]:
                    mitre_id = ref.get("external_id", "N/A")
                    break

            text_content = f"MITRE ID: {mitre_id} | Type: {obj_type} | Name: {name} | Description: {desc[:1500]}"

            doc = {
                "stix_id": obj.get("id"),
                "mitre_id": mitre_id,
                "type": obj_type,
                "name": name,
                "description": desc,
                "platforms": obj.get("x_mitre_platforms", []),
                "text_to_embed": text_content
            }

            batch.append(doc)
            if len(batch) >= BATCH_SIZE:
                batch_insert(mitre_col, batch)
                total_processed += len(batch)
                print(f"MITRE Processed: {total_processed} documents...", end="\r")
                batch = []

    # Insert remaining documents
    if batch:
        batch_insert(mitre_col, batch)
        total_processed += len(batch)

    print(f"\nCompleted MITRE ATT&CK! Total documents stored: {total_processed}")

# Execute Pipeline
process_cisa_kev()
process_mitre_stix()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 8.4 MB/s eta 0:00:00
Connecting to MongoDB Atlas...
Loading SentenceTransformer model (all-MiniLM-L6-v2)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]


--- Processing CISA KEV Feed ---

Completed CISA KEV! Total documents stored: 1638

--- Processing MITRE ATT&CK STIX Feed ---


KeyboardInterrupt: 

In [ ]:
import os
import json
import glob
import torch
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. Configuration & GPU Setup
# ---------------------------------------------------------
MONGO_URI = "mongodb+srv://@rag..mongodb.net/?appName=RAG"
SAVE_DIR = "/content/drive/MyDrive/Economic_Times/Dataset"
BATCH_SIZE = 256  # Larger batch size works well on GPU

# Detect CUDA / GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Device assigned: {device.upper()}")

print("Connecting to MongoDB Atlas...")
client = MongoClient(MONGO_URI)
db = client["cyber_intelligence_rag"]

kev_col = db["cisa_kev_embeddings"]
mitre_col = db["mitre_attack_embeddings"]

# Clear existing records to prevent duplicates
print("Cleaning old documents from collections...")
kev_col.delete_many({})
mitre_col.delete_many({})

print("Loading SentenceTransformer model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

# Batch insert helper function
def batch_insert(collection, docs_batch):
    if not docs_batch:
        return

    texts = [doc["text_to_embed"] for doc in docs_batch]

    # Generate embeddings on GPU
    embeddings = model.encode(
        texts,
        show_progress_bar=False,
        batch_size=BATCH_SIZE,
        device=device
    )

    for doc, emb in zip(docs_batch, embeddings):
        doc["embedding"] = emb.tolist()

    collection.insert_many(docs_batch)

# ---------------------------------------------------------
# 2. Process CISA KEV Data
# ---------------------------------------------------------
def process_cisa_kev():
    kev_path = os.path.join(SAVE_DIR, "KVE", "known_exploited_vulnerabilities.json")
    if not os.path.exists(kev_path):
        print(f"Skipping KEV: File not found at {kev_path}")
        return

    print("\n--- Processing CISA KEV Feed ---")
    with open(kev_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    vulnerabilities = data.get("vulnerabilities", [])
    batch = []
    total_processed = 0

    for item in vulnerabilities:
        cve_id = item.get("cveID", "N/A")
        name = item.get("vulnerabilityName", "")
        desc = item.get("shortDescription", "")
        vendor = item.get("vendorProject", "")
        product = item.get("product", "")

        text_content = f"CVE: {cve_id} | Product: {vendor} {product} | Name: {name} | Description: {desc}"

        doc = {
            "cveID": cve_id,
            "vendorProject": vendor,
            "product": product,
            "vulnerabilityName": name,
            "shortDescription": desc,
            "requiredAction": item.get("requiredAction"),
            "knownRansomwareCampaignUse": item.get("knownRansomwareCampaignUse"),
            "text_to_embed": text_content
        }

        batch.append(doc)
        if len(batch) >= BATCH_SIZE:
            batch_insert(kev_col, batch)
            total_processed += len(batch)
            print(f"KEV Processed: {total_processed} documents...", end="\r")
            batch = []

    if batch:
        batch_insert(kev_col, batch)
        total_processed += len(batch)

    print(f"\n✅ Completed CISA KEV! Total documents stored: {total_processed}")

# ---------------------------------------------------------
# 3. Process MITRE ATT&CK STIX Data (Optimized File Targeting)
# ---------------------------------------------------------
def process_mitre_stix():
    stix_dir = os.path.join(SAVE_DIR, "MITRE-STIX")
    if not os.path.exists(stix_dir):
        print(f"Skipping MITRE: Directory not found at {stix_dir}")
        return

    print("\n--- Processing MITRE ATT&CK STIX Feed ---")

    # Target the 3 main domain bundles directly to avoid opening 10,000+ small JSONs
    target_bundles = [
        os.path.join(stix_dir, "enterprise-attack", "enterprise-attack.json"),
        os.path.join(stix_dir, "mobile-attack", "mobile-attack.json"),
        os.path.join(stix_dir, "ics-attack", "ics-attack.json")
    ]

    # Fallback to search if main paths don't match exact structure
    found_bundles = [b for b in target_bundles if os.path.exists(b)]
    if not found_bundles:
        print("Main domain bundles not found directly, scanning for bundle JSONs...")
        found_bundles = [f for f in glob.glob(os.path.join(stix_dir, "*", "*.json")) if not f.endswith(".git")]

    batch = []
    total_processed = 0
    TARGET_TYPES = {"attack-pattern", "malware", "tool", "intrusion-set", "course-of-action", "x-mitre-tactic"}

    for filepath in found_bundles:
        filename = os.path.basename(filepath)
        print(f"Loading bundle: {filename}...")

        try:
            with open(filepath, "r", encoding="utf-8") as f:
                stix_data = json.load(f)
        except Exception as e:
            print(f"Error loading {filename}: {e}")
            continue

        objects = stix_data.get("objects", []) if isinstance(stix_data, dict) else []
        print(f" Found {len(objects)} STIX objects in {filename}. Processing...")

        for obj in objects:
            obj_type = obj.get("type")

            # Skip relationships, markings, or revoked techniques
            if obj_type not in TARGET_TYPES or obj.get("revoked", False):
                continue

            name = obj.get("name", "")
            desc = obj.get("description", "")
            if not desc:
                continue

            # Extract MITRE ATT&CK ID
            mitre_id = "N/A"
            for ref in obj.get("external_references", []):
                if ref.get("source_name") in ["mitre-attack", "mitre-mobile-attack", "mitre-ics-attack"]:
                    mitre_id = ref.get("external_id", "N/A")
                    break

            text_content = f"MITRE ID: {mitre_id} | Type: {obj_type} | Name: {name} | Description: {desc[:1500]}"

            doc = {
                "stix_id": obj.get("id"),
                "mitre_id": mitre_id,
                "type": obj_type,
                "name": name,
                "description": desc,
                "platforms": obj.get("x_mitre_platforms", []),
                "text_to_embed": text_content
            }

            batch.append(doc)
            if len(batch) >= BATCH_SIZE:
                batch_insert(mitre_col, batch)
                total_processed += len(batch)
                print(f" MITRE Processed: {total_processed} documents...", end="\r")
                batch = []

    if batch:
        batch_insert(mitre_col, batch)
        total_processed += len(batch)

    print(f"\n✅ Completed MITRE ATT&CK! Total documents stored: {total_processed}")

# Execute Pipeline
process_cisa_kev()
process_mitre_stix()

⚡ Device assigned: CUDA
Connecting to MongoDB Atlas...
Cleaning old documents from collections...
Loading SentenceTransformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- Processing CISA KEV Feed ---

✅ Completed CISA KEV! Total documents stored: 1638

--- Processing MITRE ATT&CK STIX Feed ---
Loading bundle: enterprise-attack.json...
 Found 25842 STIX objects in enterprise-attack.json. Processing...
Loading bundle: mobile-attack.json...
 Found 2634 STIX objects in mobile-attack.json. Processing...
Loading bundle: ics-attack.json...
 Found 2173 STIX objects in ics-attack.json. Processing...

✅ Completed MITRE ATT&CK! Total documents stored: 2527


In [ ]:
import os
import json
import torch
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. Configuration
# ---------------------------------------------------------
MONGO_URI = "mongodb+srv://@rag..mongodb.net/?appName=RAG"
SAVE_DIR = "/content/drive/MyDrive/Economic_Times/Dataset"
BATCH_SIZE = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Device assigned: {device.upper()}")

client = MongoClient(MONGO_URI)
db = client["cyber_intelligence_rag"]

capec_col = db["capec_embeddings"]
preattack_col = db["mitre_preattack_embeddings"]

# Clear existing records to prevent duplicates
capec_col.delete_many({})
preattack_col.delete_many({})

print("Loading SentenceTransformer model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def batch_insert(collection, docs_batch):
    if not docs_batch:
        return
    texts = [doc["text_to_embed"] for doc in docs_batch]
    embeddings = model.encode(texts, show_progress_bar=False, batch_size=BATCH_SIZE, device=device)
    for doc, emb in zip(docs_batch, embeddings):
        doc["embedding"] = emb.tolist()
    collection.insert_many(docs_batch)

# ---------------------------------------------------------
# 2. Process CAPEC Data
# ---------------------------------------------------------
def process_capec():
    stix_dir = os.path.join(SAVE_DIR, "MITRE-STIX", "capec")

    # Locate CAPEC JSON file
    capec_file = None
    possible_paths = [
        os.path.join(stix_dir, "2.1", "stix-capec.json"),
        os.path.join(stix_dir, "stix-capec.json")
    ]
    for p in possible_paths:
        if os.path.exists(p):
            capec_file = p
            break

    if not capec_file:
        print("Skipping CAPEC: stix-capec.json not found.")
        return

    print("\n--- Processing CAPEC Feed ---")
    with open(capec_file, "r", encoding="utf-8") as f:
        stix_data = json.load(f)

    objects = stix_data.get("objects", [])
    print(f"Found {len(objects)} objects in {os.path.basename(capec_file)}...")

    batch = []
    total_processed = 0

    for obj in objects:
        if obj.get("type") != "attack-pattern" or obj.get("revoked", False):
            continue

        name = obj.get("name", "")
        desc = obj.get("description", "")
        if not desc:
            continue

        capec_id = "N/A"
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "capec":
                capec_id = ref.get("external_id", "N/A")
                break

        text_content = f"CAPEC ID: {capec_id} | Name: {name} | Description: {desc[:1500]}"

        doc = {
            "stix_id": obj.get("id"),
            "capec_id": capec_id,
            "name": name,
            "description": desc,
            "text_to_embed": text_content
        }

        batch.append(doc)
        if len(batch) >= BATCH_SIZE:
            batch_insert(capec_col, batch)
            total_processed += len(batch)
            print(f"CAPEC Processed: {total_processed} documents...", end="\r")
            batch = []

    if batch:
        batch_insert(capec_col, batch)
        total_processed += len(batch)

    print(f"\n✅ Completed CAPEC! Total documents stored: {total_processed}")

# ---------------------------------------------------------
# 3. Process PRE-ATT&CK Data
# ---------------------------------------------------------
def process_preattack():
    preattack_file = os.path.join(SAVE_DIR, "MITRE-STIX", "pre-attack", "pre-attack.json")

    if not os.path.exists(preattack_file):
        print("Skipping PRE-ATT&CK: pre-attack.json not found.")
        return

    print("\n--- Processing PRE-ATT&CK Feed ---")
    with open(preattack_file, "r", encoding="utf-8") as f:
        stix_data = json.load(f)

    objects = stix_data.get("objects", [])
    print(f"Found {len(objects)} objects in pre-attack.json...")

    batch = []
    total_processed = 0
    TARGET_TYPES = {"attack-pattern", "intrusion-set", "tool"}

    for obj in objects:
        obj_type = obj.get("type")
        if obj_type not in TARGET_TYPES or obj.get("revoked", False):
            continue

        name = obj.get("name", "")
        desc = obj.get("description", "")
        if not desc:
            continue

        mitre_id = "N/A"
        for ref in obj.get("external_references", []):
            if ref.get("source_name") == "mitre-pre-attack":
                mitre_id = ref.get("external_id", "N/A")
                break

        text_content = f"PRE-ATT&CK ID: {mitre_id} | Type: {obj_type} | Name: {name} | Description: {desc[:1500]}"

        doc = {
            "stix_id": obj.get("id"),
            "mitre_id": mitre_id,
            "type": obj_type,
            "name": name,
            "description": desc,
            "text_to_embed": text_content
        }

        batch.append(doc)
        if len(batch) >= BATCH_SIZE:
            batch_insert(preattack_col, batch)
            total_processed += len(batch)
            print(f"PRE-ATT&CK Processed: {total_processed} documents...", end="\r")
            batch = []

    if batch:
        batch_insert(preattack_col, batch)
        total_processed += len(batch)

    print(f"\n✅ Completed PRE-ATT&CK! Total documents stored: {total_processed}")

# Execute Pipeline
process_capec()
process_preattack()

⚡ Device assigned: CUDA
Loading SentenceTransformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- Processing CAPEC Feed ---
Found 2666 objects in stix-capec.json...

✅ Completed CAPEC! Total documents stored: 613

--- Processing PRE-ATT&CK Feed ---
Found 268 objects in pre-attack.json...

✅ Completed PRE-ATT&CK! Total documents stored: 181


In [ ]:
import os
import json
import torch
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer

# ---------------------------------------------------------
# 1. Setup & Config
# ---------------------------------------------------------
MONGO_URI = "mongodb+srv://@rag..mongodb.net/?appName=RAG"
SAVE_DIR = "/content/drive/MyDrive/Economic_Times/Dataset"
CVE_DIR = os.path.join(SAVE_DIR, "CVE")  # Adjust subfolder name if needed

BATCH_SIZE = 256  # Batches sent to GPU & MongoDB

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"⚡ Device assigned: {device.upper()}")

# MongoDB Setup
client = MongoClient(MONGO_URI)
db = client["cyber_intelligence_rag"]
cve_col = db["cve_embeddings"]

# Clear collection to avoid duplicates on re-runs
print("Cleaning old records from cve_embeddings collection...")
cve_col.delete_many({})

print("Loading SentenceTransformer model (all-MiniLM-L6-v2)...")
model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

def batch_insert(collection, docs_batch):
    if not docs_batch:
        return
    texts = [doc["text_to_embed"] for doc in docs_batch]
    embeddings = model.encode(texts, show_progress_bar=False, batch_size=BATCH_SIZE, device=device)
    for doc, emb in zip(docs_batch, embeddings):
        doc["embedding"] = emb.tolist()
    collection.insert_many(docs_batch)

# ---------------------------------------------------------
# 2. CVE Record Extractor (Handles Multiple CVE Formats)
# ---------------------------------------------------------
def extract_cve_records_from_json(data):
    """
    Helper to extract (cve_id, description, text_to_embed, doc) tuples
    regardless of whether the file is a single CVE-v5 file or a bulk NVD JSON bundle.
    """
    records = []

    # Case A: NVD Bulk Feed (List under "CVE_Items" or "vulnerabilities")
    if isinstance(data, dict) and ("CVE_Items" in data or "vulnerabilities" in data):
        items = data.get("CVE_Items") or data.get("vulnerabilities") or []
        for item in items:
            cve_obj = item.get("cve", item)
            cve_id = cve_obj.get("id") or cve_obj.get("CVE_data_meta", {}).get("ID", "N/A")

            # Extract description
            descriptions = cve_obj.get("descriptions", []) or cve_obj.get("description", {}).get("description_data", [])
            desc_text = ""
            for d in descriptions:
                if d.get("lang") in ["en", "en-US"]:
                    desc_text = d.get("value", "")
                    break

            if desc_text:
                text_to_embed = f"CVE ID: {cve_id} | Description: {desc_text[:1500]}"
                records.append({
                    "cve_id": cve_id,
                    "description": desc_text,
                    "text_to_embed": text_to_embed
                })

    # Case B: Individual CVE Record (cvelistV5 format or single JSON)
    elif isinstance(data, dict) and ("cveMetadata" in data or "id" in data or "CVE_data_meta" in data):
        cve_id = data.get("cveMetadata", {}).get("cveId") or data.get("id") or "N/A"

        # Extract description from containers
        desc_text = ""
        containers = data.get("containers", {}).get("cna", {}).get("descriptions", [])
        for d in containers:
            if d.get("lang") in ["en", "en-US"]:
                desc_text = d.get("value", "")
                break

        if not desc_text and "description" in data:
            desc_text = str(data["description"])

        if desc_text:
            text_to_embed = f"CVE ID: {cve_id} | Description: {desc_text[:1500]}"
            records.append({
                "cve_id": cve_id,
                "description": desc_text,
                "text_to_embed": text_to_embed
            })

    return records

# ---------------------------------------------------------
# 3. High-Performance Recursive File Scanner
# ---------------------------------------------------------
def process_all_cve_files():
    # If the files are in SAVE_DIR directly or SAVE_DIR/CVE
    target_dir = CVE_DIR if os.path.exists(CVE_DIR) else SAVE_DIR

    print(f"\n--- Scanning and Processing CVE Files in: {target_dir} ---")

    batch = []
    total_processed = 0
    files_scanned = 0

    # Fast directory tree walk using os.walk
    for root, dirs, files in os.walk(target_dir):
        # Skip git folders if present
        if ".git" in root or ".github" in root:
            continue

        for filename in files:
            if not filename.endswith(".json"):
                continue

            filepath = os.path.join(root, filename)
            files_scanned += 1

            try:
                with open(filepath, "r", encoding="utf-8") as f:
                    stix_data = json.load(f)
            except Exception:
                continue

            extracted_docs = extract_cve_records_from_json(stix_data)

            for doc in extracted_docs:
                batch.append(doc)

                if len(batch) >= BATCH_SIZE:
                    batch_insert(cve_col, batch)
                    total_processed += len(batch)
                    print(f"Files Scanned: {files_scanned} | CVEs Stored: {total_processed}...", end="\r")
                    batch = []

    # Insert remaining buffer
    if batch:
        batch_insert(cve_col, batch)
        total_processed += len(batch)

    print(f"\n✅ Finished processing all CVE files!")
    print(f"📊 Total JSON Files Scanned: {files_scanned}")
    print(f"💾 Total CVE Documents Embedded & Stored: {total_processed}")

# Run Execution
process_all_cve_files()

⚡ Device assigned: CUDA
Cleaning old records from cve_embeddings collection...
Loading SentenceTransformer model (all-MiniLM-L6-v2)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


--- Scanning and Processing CVE Files in: /content/drive/MyDrive/Economic_Times/Dataset/CVE ---


KeyboardInterrupt: 

In [ ]:
import os
!pip install weasyprint
import weasyprint
!pip install pypdf
import pypdf

# Define HTML template matching the ET AI Hackathon 2026 Problem Statement style
html_content = """<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<style>
  @page {
    size: A4;
    margin: 12mm 12mm 12mm 12mm;
    @bottom-right {
      content: "Page " counter(page) " of " counter(pages);
      font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
      font-size: 8pt;
      color: #666;
    }
    @bottom-left {
      content: "ET AI HACKATHON 2026 — TRATA Project Submission";
      font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
      font-size: 8pt;
      color: #666;
    }
  }

  *, *::before, *::after {
    box-sizing: border-box;
  }

  body {
    font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif;
    color: #222222;
    margin: 0;
    padding: 0;
    font-size: 9.2pt;
    line-height: 1.35;
    background-color: #ffffff;
  }

  /* Top Banner Bar */
  .top-header {
    border-bottom: 2px solid #8B0000;
    padding-bottom: 6px;
    margin-bottom: 10px;
  }

  .header-table {
    width: 100%;
    border-collapse: collapse;
  }

  .header-title {
    font-size: 13pt;
    font-weight: 800;
    color: #8B0000;
    text-transform: uppercase;
    letter-spacing: 0.5px;
  }

  .header-subtitle {
    font-size: 8.5pt;
    color: #555;
    font-weight: 600;
  }

  .header-meta {
    text-align: right;
    font-size: 8pt;
    color: #444;
  }

  /* Main Title Block */
  .title-block {
    margin-bottom: 10px;
    background: #fcf8f8;
    border-left: 4px solid #8B0000;
    padding: 8px 12px;
  }

  .problem-num {
    display: inline-block;
    background: #8B0000;
    color: #ffffff;
    font-size: 14pt;
    font-weight: bold;
    padding: 2px 8px;
    border-radius: 3px;
    margin-right: 8px;
    vertical-align: middle;
  }

  .project-title {
    display: inline;
    font-size: 13.5pt;
    font-weight: bold;
    color: #111111;
    vertical-align: middle;
  }

  .theme-line {
    font-size: 8.5pt;
    color: #8B0000;
    font-weight: bold;
    margin-top: 4px;
    text-transform: uppercase;
    letter-spacing: 0.3px;
  }

  .team-info {
    font-size: 8pt;
    color: #444;
    margin-top: 3px;
  }

  /* Headings */
  h2.section-header {
    font-size: 9.5pt;
    font-weight: 800;
    color: #8B0000;
    text-transform: uppercase;
    letter-spacing: 0.5px;
    margin: 9px 0 4px 0;
    padding-bottom: 2px;
    border-bottom: 1px solid #e0e0e0;
    page-break-after: avoid;
  }

  p {
    margin: 0 0 6px 0;
    text-align: justify;
  }

  /* Bullet lists */
  ul {
    margin: 0 0 6px 0;
    padding-left: 14px;
  }

  li {
    margin-bottom: 3.5px;
    text-align: justify;
  }

  li strong {
    color: #111;
  }

  /* Custom Grid/Table Layouts */
  .two-col-table {
    width: 100%;
    border-collapse: collapse;
    margin-bottom: 6px;
  }

  .two-col-table td {
    vertical-align: top;
    width: 50%;
    padding-right: 8px;
  }

  .two-col-table td:last-child {
    padding-right: 0;
    padding-left: 8px;
  }

  /* Key-Value / Highlight Box */
  .highlight-box {
    background-color: #f8f9fa;
    border: 1px solid #e9ecef;
    border-radius: 4px;
    padding: 6px 10px;
    margin-bottom: 8px;
  }

  .table-custom {
    width: 100%;
    border-collapse: collapse;
    margin-top: 4px;
    margin-bottom: 6px;
    font-size: 8.2pt;
  }

  .table-custom th {
    background-color: #8B0000;
    color: #ffffff;
    text-align: left;
    padding: 4px 8px;
    font-weight: bold;
    font-size: 8pt;
    text-transform: uppercase;
  }

  .table-custom td {
    padding: 3.5px 8px;
    border-bottom: 1px solid #e9ecef;
  }

  .table-custom tr:nth-child(even) td {
    background-color: #fdfbfb;
  }

  .badge {
    background: #eef2f7;
    color: #1a365d;
    padding: 1px 5px;
    border-radius: 3px;
    font-family: monospace;
    font-size: 8pt;
    font-weight: bold;
  }

  .link-text {
    color: #0056b3;
    text-decoration: none;
    font-weight: 600;
  }
</style>
</head>
<body>

  <div class="top-header">
    <table class="header-table">
      <tr>
        <td>
          <div class="header-title">ET AI HACKATHON 2026</div>
          <div class="header-subtitle">Official Project Proposal & Technical Specification</div>
        </td>
        <td class="header-meta">
          <strong>Team:</strong> NotDecidedYet<br>
          <strong>Leader:</strong> Arnesh Kumar Gupta (USICT)
        </td>
      </tr>
    </table>
  </div>

  <div class="title-block">
    <span class="problem-num">7</span>
    <div class="project-title">TRATA: AI-Driven Cyber Resilience for Critical National Infrastructure</div>
    <div class="theme-line">Theme: Cybersecurity / Industrial Intelligence / National Security</div>
    <div class="team-info">
      <strong>Team Members:</strong> Arnesh Kumar Gupta (Backend/R&D), Shaurya Narian Singh (Cyber Analyst), Chinmay Kapila (ML Engineer), Akash Parashar (DevOps) | USICT, GGSIPU
    </div>
  </div>

  <h2 class="section-header">Problem Context</h2>
  <p>
    Critical national infrastructure (CNI) has become a primary target for sophisticated cyber actors, and India's public institutions have borne a severe brunt over recent years. CERT-In handled over 1.59 million cybersecurity incidents in 2023 alone, with attacks accelerating across 2024–2026. High-profile incidents—such as the two-week paralysis of AIIMS Delhi by ransomware in 2022 and recent digital infrastructure breaches at CBSE—demonstrate the vulnerability of essential public systems. India's National Cyber Security Policy acknowledges that over 70% of government and public sector entities operate on end-of-life (EOL) IT infrastructure, creating significant exposure.
  </p>
  <p>
    Traditional security relies almost exclusively on static signature-based detection (YARA hashes, known IPs, manual firewall rules). However, signature-based tools fail against low-and-slow Advanced Persistent Threats (APTs), because signatures are only generated <em>after</em> a breach has already succeeded. What public sector infrastructure requires is a behavioural intelligence layer that continuously detects subtle operational anomalies, correlates weak signals across heterogeneous IT/OT logs, and responds before systemic disruption occurs.
  </p>

  <h2 class="section-header">Challenge Statement & Solution Paradigm</h2>
  <p>
    <strong>Challenge:</strong> Build an AI-powered Cyber Resilience platform for critical national infrastructure that autonomously detects behavioural anomalies, correlates weak signals across enterprise network flows and host telemetry, maps attack progression against global threat frameworks, and orchestrates containment actions—compressing detection and response from weeks to real-time seconds.
  </p>
  <p>
    <strong>TRATA Solution (Threat Response and Tactical Assessment):</strong> TRATA replaces reactive signature detection with a multi-layered behavioural AI intelligence fabric. It fuses deep network traffic modeling, real-time graph analytics, vector-based threat intelligence, and autonomous RAG synthesis to deliver end-to-end cyber resilience across critical infrastructure nodes.
  </p>

  <h2 class="section-header">Core Machine Learning & Detection Pipeline</h2>
  <ul>
    <li><strong>Network Traffic & PCAP Inspection Engine:</strong> Captures local network packets using <span class="badge">tcpdump</span>, converts them into flow CSVs via <span class="badge">CICFlowMeter</span>, and feeds preprocessed flows into a custom 1D-CNN model trained on the benchmark <span class="badge">CSE-CIC-IDS2018</span> dataset (achieving <strong>98.4% attack detection accuracy</strong>). Malicious IPs are immediately blocked at the proxy layer.</li>
    <li><strong>Host & Behavioral Graph Analytics Engine:</strong> Ingests system logs from <span class="badge">Auditd</span> and <span class="badge">Zeek</span>, mapping user activity, running processes, and network connections into a dynamic graph structure. Utilizes <span class="badge">NetworkX</span> graph models trained on the massive 58-day enterprise <span class="badge">LANL</span> dataset (achieving <strong>97.5% anomaly accuracy</strong>) to identify low-and-slow lateral movement.</li>
    <li><strong>Unified Compiled Pattern Engine:</strong> Consolidates YARA rules into a single compiled binary ruleset (<span class="badge">core.yarc</span>) for instant file/malware string matching, complemented by portable <span class="badge">Sigma</span> rules for real-time log inspection.</li>
  </ul>

  <h2 class="section-header">Comprehensive Dataset Ecosystem & Autonomous RAG Engine</h2>
  <p>
    TRATA aggregates and synthesizes eight foundational security datasets and feeds to provide context-aware incident response:
  </p>

  <table class="two-col-table">
    <tr>
      <td>
        <ul>
          <li><strong>CSE-CIC-IDS2018:</strong> Modern network traffic dataset with diverse multi-day attack vectors.</li>
          <li><strong>LANL Dataset:</strong> Enterprise multi-log dataset (auth, process, DNS) over 58 days.</li>
          <li><strong>CVE Feeds:</strong> Standardized catalog of publicly disclosed software vulnerabilities.</li>
          <li><strong>CISA KEV:</strong> Authoritative catalog tracking actively exploited flaws in the wild.</li>
        </ul>
      </td>
      <td>
        <ul>
          <li><strong>MITRE ATT&CK / STIX:</strong> Knowledge base mapping adversary TTPs and attack patterns.</li>
          <li><strong>CERT-In Advisories:</strong> Official national vulnerability notes and remediation guidance.</li>
          <li><strong>Sigma Rules:</strong> Portable open signature format for cross-platform log queries.</li>
          <li><strong>YARA Rules:</strong> String and binary pattern matching for malware identification.</li>
        </ul>
      </td>
    </tr>
  </table>

  <p>
    <strong>Autonomous RAG Pipeline:</strong> An automated background agent continuously fetches incoming CVE, KEV, MITRE, and CERT-In updates, generating 384-dimensional dense embeddings using <span class="badge">all-MiniLM-L6-v2</span> and storing them in a <span class="badge">MongoDB Atlas Vector DB</span>. When analysts or security workflows query the system, cosine similarity retrieves relevant threat context, which a lightweight LLM (<span class="badge">google/flan-t5-small</span>) converts into actionable, natural-language threat intelligence briefs.
  </p>

  <h2 class="section-header">Implementation Cost & Infrastructure Scale</h2>
  <p>
    TRATA is engineered for fiscal efficiency and high telemetry throughput, utilizing lightweight local models (~300MB footprint) to run cost-effectively across regional public sector deployments:
  </p>

  <table class="table-custom">
    <thead>
      <tr>
        <th>Component</th>
        <th>Specification & Plan Details</th>
        <th>Monthly Cost (EST)</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td><strong>Data & Threat Feeds</strong></td>
        <td>Open Datasets (CSE-CIC-IDS2018, LANL, CVE, KEV, MITRE, CERT-In, YARA, Sigma)</td>
        <td><strong>₹0</strong> (Open Source)</td>
      </tr>
      <tr>
        <td><strong>GPU Compute</strong></td>
        <td>2–3 Virtual GPUs (NVIDIA RTX 3090 / A10 via RunPod) for low-latency vector embeddings</td>
        <td>₹20,000 – ₹33,000</td>
      </tr>
      <tr>
        <td><strong>Database & Vector Storage</strong></td>
        <td>MongoDB Atlas Dedicated Cluster (M30/M40 Tier, 1TB Storage, Vector Search Index)</td>
        <td>₹32,000 – ₹48,000</td>
      </tr>
      <tr>
        <td><strong>App & Proxy Server</strong></td>
        <td>Cloud Compute Instances (32GB RAM, multi-vGPU instances on EC2 / Vast.ai)</td>
        <td>₹25,000 – ₹42,000</td>
      </tr>
      <tr>
        <td><strong>Total Combined Cost</strong></td>
        <td><strong>Complete Operational Stack (Handles 1,500–2,500 active analysts & 8k events/sec)</strong></td>
        <td><strong>₹57,000 – ₹90,000</strong></td>
      </tr>
    </tbody>
  </table>

  <h2 class="section-header">Real-World Impact & Technical Innovations</h2>
  <ul>
    <li><strong>MTTD Compression:</strong> Reduces Mean-Time-to-Detect (MTTD) from weeks of undetected dwell time down to real-time behavioral alerts and automated proxy mitigation.</li>
    <li><strong>Full Forensic Auditability:</strong> Every automated response, IP block, proxy rule, and RAG retrieval is immutably logged for complete compliance and forensic review.</li>
    <li><strong>High-Throughput Architecture:</strong> Decoupled asynchronous telemetry ingestion pipelines handle 5,000 to 8,000 flow events per second with zero drop rates.</li>
    <li><strong>Future Roadmap:</strong> <em>Phase 1:</em> Automated SOAR playbooks for host isolation; <em>Phase 2:</em> Federated threat intelligence sharing across public sector ministries; <em>Phase 3:</em> Cyber Resilience Digital Twin for red-team attack path simulations.</li>
  </ul>

  <h2 class="section-header">Project Deliverables & Repository Links</h2>
  <table class="two-col-table">
    <tr>
      <td>
        <ul>
          <li><strong>Working Prototype:</strong> Full-stack detection & proxy platform</li>
          <li><strong>Architecture Diagram:</strong> ML & RAG pipeline schema</li>
          <li><strong>Live Frontend:</strong> <span class="link-text">detector-trata.vercel.app</span></li>
        </ul>
      </td>
      <td>
        <ul>
          <li><strong>GitHub Repository:</strong> <span class="link-text">github.com/Arnesh0512/TRATA</span></li>
          <li><strong>Live Backend API:</strong> <span class="link-text">trata-production.up.railway.app/docs</span></li>
          <li><strong>Presentation & Video Demo:</strong> Submitted with project package</li>
        </ul>
      </td>
    </tr>
  </table>

  <h2 class="section-header">Judging Criteria Alignment</h2>
  <table class="table-custom">
    <thead>
      <tr>
        <th>Criteria</th>
        <th>Weight</th>
        <th>TRATA Technical Alignment & Implementation</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td><strong>Innovation</strong></td>
        <td>25%</td>
        <td>Multi-modal fusion of 1D-CNN packet analysis, NetworkX graph behavioral tracking, and autonomous RAG threat synthesis.</td>
      </tr>
      <tr>
        <td><strong>Business Impact</strong></td>
        <td>25%</td>
        <td>Protects government EOL IT infrastructure, preventing catastrophic downtime (e.g., AIIMS/CBSE scale) at under ₹90k/mo.</td>
      </tr>
      <tr>
        <td><strong>Technical Excellence</strong></td>
        <td>20%</td>
        <td>98.4% CNN detection accuracy, 97.5% graph anomaly precision, compiled YARA rules (<span class="badge">core.yarc</span>), and 384-dim MongoDB vector search.</td>
      </tr>
      <tr>
        <td><strong>Scalability</strong></td>
        <td>15%</td>
        <td>Asynchronous pipeline scaling to 8,000 events/sec and 2,500 concurrent analyst sessions with tiny model footprint (~300MB).</td>
      </tr>
      <tr>
        <td><strong>User Experience</strong></td>
        <td>15%</td>
        <td>Interactive dashboard showing live threat alerts, automated proxy blocking, and natural-language RAG intelligence query interface.</td>
      </tr>
    </tbody>
  </table>

</body>
</html>
"""

# Render PDF using WeasyPrint
pdf_filename = "TRATA_ET_AI_Hackathon_2026_Proposal.pdf"
weasyprint.HTML(string=html_content).write_pdf(pdf_filename)

# Check page count
reader = pypdf.PdfReader(pdf_filename)
print(f"Generated PDF Page Count: {len(reader.pages)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 101.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.4/829.4 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 26.5 MB/s eta 0:00:00


Generated PDF Page Count: 2
